In [1]:
import io
import re
import requests
import pdfplumber
import pytesseract
import psycopg2 as pg
from pathlib import Path
from pdf2image import convert_from_bytes
import os
from dotenv import load_dotenv

In [44]:
# Save the link to access the PDF
url = "https://s3-fs1.fsn1.your-objectstorage.com/s3-fs1/sign_documents/205049000036322486.pdf"

In [45]:
# Return the response of te link, 
# 200: meaning it can be access
# 404: failed to access the PDF
response = requests.get(url)

In [46]:
# Virtual PDF file - exists only in RAM, not on disk.
# Faster access from RAM memory than disk memory.
# When the program is ended, the memory is released.
pdf_bytes = io.BytesIO(response.content)

In [47]:
# Open the virtual PDF saved in RAM memory as pdf,
# with pdfplumber: closes the virtual file when leaving the notebook block 
with pdfplumber.open(pdf_bytes) as pdf:
    # enumerate(): add a numeric counter in each iteration,
    # where page is an object from pdfplumber representing the loaded page,
    # ready to be manipulated
    for i, page in enumerate(pdf.pages[:2]):
        # Extract the text from the pdfplumber object (reprenting the loaded page)
        text = page.extract_text()
        # Print the numeric counter, which represent each page
        print(i)
        # Print the text from the page
        print(text)

0
Zoho Sign Document ID: 2BF51FA9-BVPDQPX-A6KOC74I5ESJKPFCE7IX_B4KFMB0KUXO_QG
RMA FORM
To:
P:
E: Warranty Serial #: Date Order #
A:
Account
Item Purchased VIN For Vehicle
Manager
Problem(s) experienced with the ORIGINAL MODULE
ORIGINAL MODULE's Symptoms (e.g: vehicle misfiring, stalling, shutting off):
Was the vehicle starting with the ORIGINAL MODULE?
IF NOT, was the engine receiving any of the following:
YES NO
Fuel Spark Starter Activating
Error codes with the ORIGINAL MODULE (error code number, e.g: P1000, P0300):
Steps taken to diagnose the problem with the ORIGINAL MODULE:
Result from those Steps:
Page 1 of 3
accounts@fs1inc.com 19 Wilbur Street,
(516) 766-2223
www.fs1inc.com Lynbrook, NY, 11563
1
Zoho Sign Document ID: 2BF51FA9-BVPDQPX-A6KOC74I5ESJKPFCE7IX_B4KFMB0KUXO_QG
RMA FORM
To:
P:
E: Warranty Serial #: Date Order #
A:
Account
Item Purchased VIN For Vehicle
Manager
Problem(s) experienced with the unit from FLAGSHIP ONE
FLAGSHIP ONE MODULE's Symptoms (e.g: vehicle misfiring,

## Page 1

In [99]:
# Return the internal cursor to the beginning of the file.
# When reading the file above, the internal cursor is moved until the end,
# seek() return it to the beginning of the file, so that the next reading starts from the beginning of the file
pdf_bytes.seek(0)

# Convert from BytesIO object to PIL.Image of page 1
# dpi (Dots Per Inch): pixels density per inch when converting from image to PDF.
# As higher as dpi is, better is the image quality
images_page1 = convert_from_bytes(pdf_bytes.read(), first_page=1, last_page=1, dpi=600)

In [100]:
# Convert from PIL.Image object to text/string
ocr_text_page1 = pytesseract.image_to_string(images_page1[0])

In [101]:
ocr_text_page1

"Zoho Sign Document ID: 2BF51FA9-BVPDQPX-A6KOC/74I5ESJKPFCE7IX_B4KFMBOKUXO_QG\n\nNY FLAGSHIP\n\nTo: H10050450 Daniel Canales\n\nRMA FORM\n\nP: 6317039590\n\nE: dancanales87@gmail.com Warranty Serial #: Date Order #\nA: 241 38th St, Lindenhurst, 11757, United States 1099897 2026-04-28 Phone\nAccount .\nItem Purchased | VIN For Vehicle\nManager\nFaith L 68153847AH | 2013 Dodge Ram Truck 3.6L ECM Engine Computer PCM ECU 1C6RR7GGXDS598785\n\nProgrammed Plug&Play | 05150678AC\n\nORIGINAL MODULE's Symptoms (e.g: vehicle misfiring, stalling, shutting off):\n\nMy ecu had aa short circuit so | could not access it anymore needed to be fully replaced\n\nWas the vehicle starting with the ORIGINAL MODULE?\n\nIF NOT, was the engine receiving any of the following:\n\n[ | NO [ ] Fuel [ ] Spark [] Starter Activating\n\nError codes with the ORIGINAL MODULE (error code number, e.g: P1000, P0300):\n\nU0100\n\nSteps taken to diagnose the problem with the ORIGINAL MODULE:\n\nGet new\n\nResult from those Ste

In [102]:
# Seach for the automake name in the page 1
automake_match = re.search(r'VIN For Vehicle[^\n]*\n.*?\d{4}(?!-)\s+([A-Za-z]+)', ocr_text_page1, re.DOTALL)

In [103]:
# Select the 2nd group on the string where the automaker name is located 
make_name = automake_match.group(1)

In [104]:
# re.search: search for the pattern in the text
# :\n\n: 
# (.*?): capture all text after \n\n until the next \n\n
# (?=\n\n|\Z): regex to limit where the search stops, next blank line
orig_dtc_field_string = re.search(r'Error codes with the ORIGINAL MODULE[^\n]*:\n\n(.*?)(?=\n\n|\Z)', ocr_text_page1, re.DOTALL)

In [110]:
orig_dtc_field = orig_dtc_field_string.group(1).strip()

In [111]:
orig_dtc_field

'U0100'

In [107]:
# Find all dtcs in the string
orig_dtcs_match = re.findall(r'[PCBU][0-9A-F]{4}', orig_dtc_field, re.IGNORECASE)

In [113]:
if orig_dtcs_match:
    orig_dtcs = orig_dtcs_match
    print(orig_dtcs)
else:
    orig_dtcs = orig_dtc_field
    print(orig_dtcs)

['U0100']


## Page 2

In [65]:
# Return the internal cursor to the beginning of the file.
# When reading the file above, the internal cursor is moved until the end,
# seek() return it to the beginning of the file, so that the next reading starts from the beginning of the file
pdf_bytes.seek(0)

0

In [66]:
# Convert from BytesIO object to PIL.Image
images_page2 = convert_from_bytes(pdf_bytes.read(), first_page=2, last_page=2, dpi=600)

In [67]:
# Convert from PIL.Image to string/text
ocr_text_page2 = pytesseract.image_to_string(images_page2[0])

In [68]:
ocr_text_page2

"Zoho Sign Document ID: 2BF51FA9-BVPDQPX-A6KOC/74I5ESJKPFCE7IX_B4KFMBOKUXO_QG\n\nNY FLAGSHIP\n\nTo: H10050450 Daniel Canales\n\nRMA FORM\n\nP: 6317039590\nE: dancanales87@gmail.com\n\nA: 241 38th St, Lindenhurst, 11757, United States 1099897 2026-04-28 Phone\nAccount .\nItem Purchased | VIN For Vehicle\nManager\nFaith L 68153847AH | 2013 Dodge Ram Truck 3.6L ECM Engine Computer PCM ECU Programmed Plug&Play | 1C6RR7GGXDS598785\n05150678AC\n\nFLAGSHIP ONE MODULE's Symptoms (e.g: vehicle misfiring, stalling, shutting off):\n\nMy car had an aftermarket remote start when | would press the lock button 3times the car would start, since replacing the new module ECU that feature stopped working so my\nquestion is does that need to be programmed into the ecu to make it work again?\n\nDoes the vehicle start with the FLAGSHIP ONE MODULE?\n\nIF NOT, was the engine receiving any of the following:\n\nNO\nL | [ ] Fuel [] Spark [ ] Starter Activating\n\nError codes with FLAGSHIP ONE MODULE (Error code 

In [119]:
# re.search: search for the pattern in the text
# (.*?): capture all text after \n\n until the next \n\n
# (?=\n\n|\Z): regex to limit where the search stops, next blank line
fs1_dtc_field = re.search(r'Error codes with FLAGSHIP ONE MODULE[^\n]*:\n\n(.*?)(?=\n\n|\Z)', ocr_text_page2, re.DOTALL)

In [120]:
fs1_dtc_fil_raw_string = fs1_dtc_field.group(1).strip()

In [121]:
# Return a list with all codes captured based on the regex pattern 
fs1_dtc_match = re.findall(r'[PCBU][0-9A-F]{4}', fs1_dtc_fil_raw_string, re.IGNORECASE)

In [122]:
if fs1_dtc_match:
    fs1_dtcs = fs1_dtc_match
else:
    fs1_dtcs = fs1_dtc_fil_raw_string

In [123]:
fs1_dtcs

'None'

## Show dtc description

In [158]:
# Create the connection with the database from the env vars
conn = pg.connect(
    host=os.getenv("HOST_NAME"),
    port=os.getenv("PORT_NUMBER"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("USER_NAME"),
    password=os.getenv("PASSWORD")
)

In [159]:
# Read variavles from a .env files and sets them in os.environ.
# Path.cwd(): standard Python library used to read the relative path from .env 
load_dotenv(Path.cwd().parent / ".env", override=True)

True

In [160]:
# psycopg2 object responsible for execute queries
cur = conn.cursor()

In [161]:
# Create a list with the file names containing the dtcs
automaker_db_tables_names_dict = {
    "acura_dtcs": "Acura",
    "audi_dtcs": "Audi",
    "bmw_dtcs": "BMW",
    "buick_dtcs": "Buick",
    "cadillac_dtcs": "Cadillac",
    "chevrolet_dtcs": "Chevrolet",
    "chrysler_dtcs": "Chrysler",
    "dodge_dtcs": "Dodge",
    "ford_dtcs": "Ford",
    "generic_dtcs": "Generic",
    "geo_dtcs": "Geo",
    "gmc_dtcs": "GMC",
    "honda_dtcs": "Honda",
    "hyundai_dtcs": "Hyundai",
    "hummer_dtcs": "Hummer",
    "infiniti_dtcs": "Infiniti",
    "isuzu_dtcs": "Isuzu",
    "jaguar_dtcs": "Jaguar",
    "jeep_dtcs": "Jeep",
    "kia_dtcs": "Kia",
    "land_rover_dtcs": "Land Rover",
    "lexus_dtcs": "Lexus",
    "mazda_dtcs": "Mazda",
    "mercedes_benz_dtcs": "Mercedes-Benz",
    "mini_dtcs": "Mini",
    "mitsubishi_dtcs": "Mitsubishi",
    "nissan_dtcs": "Nissan",
    "oldsmobile_dtcs": "Oldsmobile",
    "pontiac_dtcs": "Pontiac",
    "saturn_dtcs": "Saturn",
    "subaru_dtcs": "Subaru",
    "toyota_dtcs": "Toyota",
    "volkswagen_dtcs": "Volkswagen"
}

In [162]:
type(orig_dtcs)

list

In [163]:
type(fs1_dtcs)

str

In [164]:
# Condition to confirm wheter the result is a string or not
if isinstance(orig_dtcs, str):
    print(orig_dtcs)
else:
    print(orig_dtcs)

['U0100']


In [165]:
# Condition to confirm wheter the result is a string or not
if isinstance(fs1_dtcs, str):
    print(fs1_dtcs)
else:
    print(fs1_dtcs)

None


In [166]:
automaker_table = None

# Loop to iterate over the dict with the table names and automakers
for table_name, maker_name in automaker_db_tables_names_dict.items():
    # Condition using case insensitive to confirm when the automaker matches
    # with the name in the dictionary
    if maker_name.lower() == make_name.lower():
        # Assign the table name to a variable 
        automaker_table = table_name
        break

In [167]:
# List to append the dtcs got from the db 
dtc_descriptions_list = []

# Iterate over the dtcs extracted from the PDF form to fetch their descriptions
for dtc_code in orig_dtcs:
    
    # Reset the description variable to avoid this variable
    # inherit the description from the last dtc 
    description = None

    # Search in the automaker's table
    if automaker_table:
        # Send the query to PostgreSQL to search for the DTC description
        # %s: psycopg2 placeholder used to replace the DTC in Python,
        # equivalent to executing from LOWER(%s) to LOWER('P0420').
        cur.execute(
            f"SELECT description FROM {automaker_table} WHERE LOWER(code) = LOWER(%s)",
            (dtc_code,)
        )
        # Retrieve the first row returned by the query,
        # returning a tuple with the results
        db_result = cur.fetchone()
        # Condition to confirm if the dtc description was fetched
        if db_result:
            # Assign the description value (first column of the returned row, which is tuple) to the variable
            description = db_result[0]

    # Fallback: search in generic_dtcs
    if not description:
        cur.execute(
            "SELECT description FROM generic_dtcs WHERE LOWER(code) = LOWER(%s)",
            (dtc_code,)
        )
        db_result = cur.fetchone()
        if db_result:
            description = db_result[0]
    
    # NOT FOUND MESSAGE if not found in any table
    if not description:
        description = "DTC NOT IN THE DATABASE"

    # Append the result to a list
    dtc_descriptions_list.append({"code": dtc_code, "description": description})

# VERY IMPORTANT: always close the database connection
cur.close()
conn.close()

In [168]:
# Iterate over the dtcs description list obtained from the db
for item in dtc_descriptions_list:
    print(f"{item["code"]} {item["description"]}")

U0100 DTC NOT IN THE DATABASE


In [155]:
if isinstance(dtc_descriptions_list, str):
    for d in dtc_descriptions_list:
        orig_dtc_lines = "\n".join(f"{d['code']} {d['description']}")
else:
    orig_dtc_lines = dtc_descriptions_list

In [156]:
orig_dtc_lines

[{'code': 'N', 'description': '(not a valid dtc)'},
 {'code': 'o', 'description': '(not a valid dtc)'},
 {'code': 'n', 'description': '(not a valid dtc)'},
 {'code': 'e', 'description': '(not a valid dtc)'}]